# RAG Memory: Separate Conversation, Checkpoint, and Long-Term State

| Field | Value |
|---|---|
| Stage | Stateful RAG |
| Difficulty | Advanced |
| Status | Complete |
| Requires network/API | No |
| Last reviewed | 2026-09-25 |

Callout - Key idea:
Memory is not one bucket: conversational context, workflow checkpoints, and long-term user data have different lifetimes and privacy contracts.

## 30-Second Summary

A pronoun follow-up uses session memory, a workflow resumes from a checkpoint, and an opt-in preference expires from long-term memory at its TTL.

## Why This Matters

Mixing all state into the prompt leaks data, prevents reliable resume, and makes deletion or expiry impossible to reason about.

## Scope

| Covers | Does not cover |
|---|---|
| Three memory classes, stable keys, checkpoint resume, consent, TTL/expiry | Vector memory store, encryption implementation, cross-device sync |


## Mental Model

```text
session turns -> resolve current conversation
checkpoint -> resume workflow exactly
long-term profile -> opt-in fact with TTL/delete
```


In [1]:
from datetime import date

session_id = "session-demo"
conversation = [{"role": "user", "text": "Tell me about Atlas retention."}, {"role": "assistant", "entity": "Atlas", "text": "Atlas retains logs for 30 days."}]
checkpoint = {"thread_id": "thread-demo", "next_step": "synthesize", "evidence_ids": ["atlas-policy"], "status": "paused"}
profile = {"user_id": "user-demo", "fact": "prefers concise answers", "consent": True, "expires_on": date(2026, 10, 1)}
TODAY = date(2026, 9, 25)


## How It Works

Session context resolves references only within the current conversation. Checkpoints store control state for deterministic resume. Long-term facts require consent, minimal scope, an expiry date, and a deletion path.


## Baseline

A single untyped memory list cannot distinguish what belongs in a prompt, what resumes execution, or what must expire.


In [2]:
baseline_memory = conversation + [checkpoint, profile]
baseline_types_declared = all("memory_type" in item for item in baseline_memory)
baseline_types_declared


False

## Technique Implementation

Typed accessors expose only the memory class required for the current operation.


In [3]:
def resolve_followup(turns: list[dict], question: str) -> str:
    entity = next(turn["entity"] for turn in reversed(turns) if "entity" in turn)
    return question.replace("it", entity)

def resume(saved: dict) -> dict:
    assert saved["status"] == "paused"
    return {**saved, "status": "running", "resumed_at": saved["next_step"]}

def read_profile(record: dict, as_of: date) -> str | None:
    return record["fact"] if record["consent"] and as_of < record["expires_on"] else None

resolved = resolve_followup(conversation, "How long does it retain logs?")
resumed = resume(checkpoint)
preference_now = read_profile(profile, TODAY)
resolved, resumed, preference_now


('How long does Atlas retain logs?',
 {'thread_id': 'thread-demo',
  'next_step': 'synthesize',
  'evidence_ids': ['atlas-policy'],
  'status': 'running',
  'resumed_at': 'synthesize'},
 'prefers concise answers')

## Controlled Experiment

We verify session resolution, exact checkpoint resume, long-term access before expiry, and automatic denial at the boundary.


In [4]:
preference_at_expiry = read_profile(profile, profile["expires_on"])
deleted_profile = {**profile, "consent": False}
preference_after_delete = read_profile(deleted_profile, TODAY)
experiment = {"resolved": resolved, "resumed_at": resumed["resumed_at"], "available_now": preference_now, "at_expiry": preference_at_expiry, "after_delete": preference_after_delete}
experiment


{'resolved': 'How long does Atlas retain logs?',
 'resumed_at': 'synthesize',
 'available_now': 'prefers concise answers',
 'at_expiry': None,
 'after_delete': None}

## Evaluation

Session memory resolves `it` to `Atlas`; the checkpoint resumes at `synthesize`; the profile is readable before **2026-10-01** and unavailable at expiry or after consent deletion.


In [5]:
assert not baseline_types_declared
assert experiment["resolved"] == "How long does Atlas retain logs?"
assert experiment["resumed_at"] == "synthesize" and experiment["available_now"] == "prefers concise answers"
assert experiment["at_expiry"] is None and experiment["after_delete"] is None
print("RAG-memory checks passed.")


RAG-memory checks passed.


## Decision Guide

| State | Store/lifetime |
|---|---|
| Current dialogue reference | Session/conversation |
| Workflow resume point | Checkpoint by thread |
| Reusable user preference | Opt-in long-term with TTL |
| Sensitive unnecessary data | Do not store |


## Failure Modes and Debugging

| Symptom | Cause | Fix |
|---|---|---|
| Cross-user leak | Weak keys | Tenant/user/session isolation |
| Wrong resume | Prompt summary used as checkpoint | Typed workflow state |
| Stale preference | No TTL/version | Expiry and refresh |
| Impossible deletion | No provenance/index | Stable record IDs and delete audit |


## Production Notes

### Observability
Log memory type, key, read/write purpose, consent, TTL, version, deletion, and resume point—never raw secrets.

### Safety and Guardrails
Minimize stored data, isolate tenants, encrypt, authorize reads, and support export/delete.

### Latency and Cost
Retrieve only memory needed for the current step and summarize bounded session windows.


## Practice

Add a second session and prove its pronoun resolution cannot read the first session's entity.

## Recall

Toggle - Recall: Why are checkpoints not conversation memory?
They preserve executable workflow state, not just dialogue meaning.

Toggle - Recall: What must long-term memory have?
Purpose, consent, identity, TTL/version, access control, and deletion.

## Sources

- [LangGraph persistence](https://docs.langchain.com/oss/python/langgraph/persistence)
- Repository-owned synthetic memory fixture

## Review Log

| Date | Status | Confidence | Next review focus |
|---|---|---|---|
| 2026-09-25 | Complete; executed and visually reviewed | High for lifecycle separation and boundary checks | Add tenant isolation and encrypted-store integration |
